# 🧪 Molecules as Bitstrings

### *A Qiskit Fall Fest lab challenge celebrating 10 years of IBM Quantum in the cloud* 🎉

In May 2016, IBM put the first real quantum computer on the cloud. A decade later,
this notebook turns the SQD idea toward its real target: molecular chemistry.

**SQD from scratch, core notebook 2 of 3** · ⏱ about 60 minutes

In notebook 1 you learned the complete SQD procedure on a toy magnet: *sample
bitstrings, crop the Hamiltonian, diagonalize the small block.* Now we apply the same
procedure to the real target: **molecules**.

**🎯 By the end you will be able to:**
- read and write electron configurations as bitstrings,
- explain why supplied full bitstrings, alpha strings, beta strings, and determinant
  subspace dimension are four different counts,
- reproduce Hartree-Fock from one determinant in the Hartree-Fock orbital basis,
- use `solve_fermion` from the SQD addon to diagonalize in any subspace you choose,
- watch a tiny, well-chosen determinant product space improve a bond-breaking curve
  that Hartree-Fock ruins.

**You should already know:** notebook 1 (concentration, cropping, variational upper bounds).
Zero chemistry background is assumed; we build what we need.

Several things change at once in this notebook, so here is the dictionary up front:

| Notebook 1 | Notebook 2 |
|---|---|
| A chain of magnets | A molecule (H₂) |
| An arrangement of 8 magnets | A seating chart of electrons (a **configuration**) |
| Bitstring = the measured state of qubits | Bitstring = a *label* for a configuration |
| $H$: a table built from two rules | $H$: a table built from `hcore` and `eri` (section 2) |
| Crop + `np.linalg.eigh` | `solve_fermion` (section 3) |
| Sample from a circuit | Postponed to notebook 3 |

The last row is worth stating explicitly: **nothing quantum happens in this notebook.**
Today we learn the new basis and the classical half of SQD; the quantum sampler
returns in notebook 3.

In [ ]:
# If you are on Google Colab or a fresh environment, uncomment and run:
# %pip install "numpy>=2.3.2" "qiskit-addon-sqd==0.12.1" "pyscf==2.14.0" "matplotlib==3.11.1"

import itertools

import numpy as np
import matplotlib.pyplot as plt
import pyscf
import pyscf.ao2mo
import pyscf.fci
from qiskit_addon_sqd.fermion import bitstring_matrix_to_ci_strs, solve_fermion

rng = np.random.default_rng(2026)

## 1 · Electrons in seats

Here's all the chemistry vocabulary we need:

- A molecule offers a ladder of **orbitals**: energy levels where electrons can sit.
  Physically, an orbital is a standing-wave cloud around the nuclei; the lower rungs
  of the ladder are clouds that hug the nuclei more tightly.
- Each orbital has exactly **two seats**, and the reason is the **Pauli exclusion
  principle**: no two electrons may occupy the same state completely. Spin is the one
  label that lets a pair share an orbital, so each orbital seats one spin-up electron
  (called **alpha**, $\alpha$) and one spin-down electron (**beta**, $\beta$).
- A **configuration** is one specific seating chart: which orbitals hold which
  electrons. We track the alpha seats and the beta seats separately because their
  counts are conserved separately: H₂'s ground state keeps exactly one alpha and one
  beta electron, and "valid configuration" will always mean "correct count in each
  half". (Hardware noise, in notebook 3, breaks exactly this.)

And the central connection to everything you did in notebook 1:

> **A configuration is just a bitstring.** One bit per orbital per spin:
> `1` = seat taken, `0` = seat empty.

Our molecule is **H₂**: two protons, two electrons. We will describe it using
4 orbitals, and that number is worth examining, because it is not a property of the
molecule itself. It is a modeling choice, set by the **basis set**: the fixed
collection of orbital shapes the calculation may combine when building its
description of the electrons. Concretely, our basis set is **6-31G**, a small
standard one that places two cloud shapes on each hydrogen atom: one compact, one
more spread out. That gives four building blocks in total, and the calculation mixes
them into 4 molecule-wide orbitals: the four rungs of the energy ladder you will see
drawn in the next cell, ordered from lowest energy upward. A larger basis set
supplies more building blocks, giving more accurate energies at the cost of more
orbitals (and, in notebook 3, more qubits).

Now the bookkeeping. Four orbitals, each offering one alpha seat and one beta seat,
makes 8 seats in total. A configuration is therefore 8 bits, one per seat: 4 alpha
bits plus 4 beta bits. We'll write each half like a Qiskit bitstring: **orbital 0 is
the rightmost bit**. The SQD addon packs the two halves into one row as
`beta_bits + alpha_bits`; the `config` helper below is the only place that packing
ever happens, so you will never assemble a packed row by hand:

In [ ]:
def config(alpha: str, beta: str) -> np.ndarray:
    """One configuration as a boolean row for the SQD addon.

    `alpha` and `beta` are strings like '0011' (orbital 0 = rightmost bit).
    The addon convention packs them as beta_bits + alpha_bits.
    """
    return np.array([c == "1" for c in beta + alpha])


def draw_configuration(alpha, beta, ax, title=""):
    """Orbital-ladder picture of a configuration."""
    norb = len(alpha)
    for i in range(norb):  # orbital i, reading each half right-to-left
        ax.hlines(i, -0.3, 0.3, color="black", lw=1.5)
        if alpha[norb - 1 - i] == "1":
            ax.annotate("", (-0.1, i + 0.38), (-0.1, i + 0.02),
                        arrowprops=dict(arrowstyle="-|>", color="#4589ff", lw=2.5))
        if beta[norb - 1 - i] == "1":
            ax.annotate("", (0.1, i + 0.02), (0.1, i + 0.38),
                        arrowprops=dict(arrowstyle="-|>", color="#fa4d56", lw=2.5))
    ax.set_xlim(-0.8, 0.8), ax.set_ylim(-0.5, norb)
    ax.set_yticks(range(norb)), ax.set_yticklabels([f"orbital {i}" for i in range(norb)])
    ax.set_xticks([])
    for side in ["top", "right", "bottom", "left"]:
        ax.spines[side].set_visible(False)
    ax.set_title(title + f"\nα={alpha}  β={beta}", family="monospace", fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
draw_configuration("0001", "0001", axes[0], "ground-floor seating (Hartree-Fock)")
draw_configuration("0010", "0010", axes[1], "both electrons bumped up")
plt.tight_layout()
plt.show()

> **Two meanings of “basis.”** The **orbital basis set** (`6-31G`) is the collection of
> spatial cloud shapes used to build molecular orbitals. The **determinant basis** is
> the collection of electron-occupation configurations used to build the many-electron
> state. Notebook 1's computational basis states are analogous to the determinant
> basis, not to the orbital basis set.

<details>
<summary>🔎 <b>Optional intuition: why “determinant”?</b></summary>

A many-electron basis state is written mathematically as a **Slater determinant**.
Exchanging two electrons swaps two rows or columns, which changes the determinant's
sign. That sign change is exactly the antisymmetry required for identical fermions
such as electrons. The bitstring is a compact label for which orbitals appear in
that determinant.

</details>

Keep this bit-order map in view. It is the convention used by `config`, Qiskit count
strings, and the SQD addon:

```text
packed row, displayed left to right:  β3 β2 β1 β0 | α3 α2 α1 α0
NumPy column index:                     0  1  2  3 |  4  5  6  7
orbital index represented:              3  2  1  0 |  3  2  1  0
                                                      orbital 0 is rightmost in each half
```

In [ ]:
def unpack_config(row):
    """Return (alpha_string, beta_string) from one packed boolean row."""
    row = np.asarray(row, dtype=bool)
    assert row.ndim == 1 and len(row) % 2 == 0, "a packed configuration must have 2*norb bits"
    norb = len(row) // 2
    beta = "".join("1" if bit else "0" for bit in row[:norb])
    alpha = "".join("1" if bit else "0" for bit in row[norb:])
    return alpha, beta


def validate_configuration(row, nelec=(1, 1), verbose=True):
    """Check the separate alpha and beta electron counts."""
    alpha, beta = unpack_config(row)
    counts = (alpha.count("1"), beta.count("1"))
    valid = counts == tuple(nelec)
    if verbose:
        mark = "✅" if valid else "❌"
        print(f"{mark} α={alpha} ({counts[0]} e-)  β={beta} ({counts[1]} e-)  target={tuple(nelec)}")
    return valid


def subspace_summary(rows, *, open_shell=True, state=None, label="subspace"):
    """Display the three input counts and the determinant product-space dimension."""
    rows = np.atleast_2d(np.asarray(rows, dtype=bool))
    ci_alpha, ci_beta = bitstring_matrix_to_ci_strs(rows, open_shell=open_shell)
    norb = rows.shape[1] // 2
    fmt = lambda values: [f"{int(value):0{norb}b}" for value in values]
    nominal_shape = (len(ci_alpha), len(ci_beta))
    print(label)
    print(f"  full sample rows supplied:       {len(rows)}")
    print(f"  unique full sample rows:         {len(np.unique(rows, axis=0))}")
    print(f"  unique alpha strings ({nominal_shape[0]}): {fmt(ci_alpha)}")
    print(f"  unique beta strings  ({nominal_shape[1]}): {fmt(ci_beta)}")
    print(f"  determinant product dimension:   {nominal_shape[0]} x {nominal_shape[1]} = "
          f"{nominal_shape[0] * nominal_shape[1]}")
    if state is not None:
        actual_shape = tuple(int(x) for x in state.amplitudes.shape)
        print(f"  solver's actual state shape:     {actual_shape} = {int(np.prod(actual_shape))} determinants")
    return {"ci_alpha": ci_alpha, "ci_beta": ci_beta, "shape": nominal_shape}


example = config("0010", "0001")
validate_configuration(example)

**🧠 Checkpoint 1.** (a) Write the alpha and beta strings for "the alpha electron moves up
to orbital 1, the beta electron stays in orbital 0". (b) H₂ must always have exactly one
alpha and one beta electron in these 4 orbitals. How many *valid* configurations are
there, out of all $2^8 = 256$ bitstrings?

<details>
<summary>💡 <b>Check your answer</b></summary>

(a) `alpha = "0010"`, `beta = "0001"` (packed row: `00010010`).

(b) The alpha electron picks 1 seat out of 4, and so does the beta electron:
$4 \times 4 = 16$ valid configurations, about 6% of all 256 bitstrings. The other 94%
carry either the wrong number of electrons or the wrong balance of alpha and beta;
they describe a charged or spin-flipped variant of H₂ rather than the neutral
molecule we are solving, so they play no part in our search. Remember this number:
in notebook 3, hardware noise will hand us exactly these out-of-balance bitstrings.

</details>

## 2 · One cell of classical chemistry (no need to memorize)

To crop-and-diagonalize we need the molecule's Hamiltonian. In notebook 1, $H$ was a
table of numbers small enough to build outright. For a molecule nobody stores the whole
table; instead, chemistry packages hand you a compact **kit of numbers** from which any
entry $\langle \text{config}_i | H | \text{config}_j \rangle$ can be assembled on
demand. The classical package `pyscf` computes that kit from just the atom positions
and a basis-set name:

- `hcore[p, q]`: the **one-electron numbers**,
- `eri[p, q, r, s]`: the **two-electron numbers**,
- `e_core`: a constant energy offset from the nuclei and, later, any frozen electrons.

Those are the only three objects needed for the first solve. The helper below also
computes reference energies and the exact tiny-system state so we can grade the lesson;
we will name those quantities after the first successful diagonalization.

In [ ]:
def h2_molecule(distance):
    """All the ingredients for H2 at the given bond distance (in Angstrom)."""
    mol = pyscf.gto.Mole()
    mol.build(atom=[["H", (0, 0, 0)], ["H", (distance, 0, 0)]], basis="6-31g", verbose=0)
    scf = pyscf.scf.RHF(mol).run()
    norb = mol.nao_nr()
    mo = scf.mo_coeff
    hcore = mo.T @ scf.get_hcore() @ mo                      # one-electron integrals
    eri = pyscf.ao2mo.restore(1, pyscf.ao2mo.kernel(mol, mo), norb)  # two-electron integrals
    e_core = mol.energy_nuc()
    e_fci, civec = pyscf.fci.FCI(mol, mo).kernel()
    return scf.e_tot, hcore, eri, e_core, e_fci, np.asarray(civec)


e_hf, hcore, eri, e_core, e_fci, civec = h2_molecule(0.74)
print("hcore shape:", hcore.shape)
print("eri shape:  ", eri.shape)
print(f"constant e_core: {e_core:.6f} Ha")

## 3 · `solve_fermion`: notebook 1's crop, molecule edition

The SQD addon's `solve_fermion` does exactly what your `subspace_energy` did, but for
electrons: give it configuration rows plus the integrals, and it returns the best
energy achievable inside the resulting determinant subspace.

There is one important change from notebook 1. The solver separates each full row into
an alpha half and a beta half, then uses **every alpha/beta pairing**. If the retained
sets have $M$ alpha strings and $N$ beta strings, the actual determinant basis has
$M \times N$ members.

> **Indistinguishability note.** Electrons are not individually tagged. A configuration
> records which alpha and beta seats are occupied, so “both electrons jump from orbital
> 0 to orbital 1” is shorthand for an occupancy change. It does not mean we can track a
> named “electron 1” from one seat to another.

For the main lesson we pass `open_shell=True`. That keeps the alpha and beta string
sets separate, so the product space follows directly from the rows we supplied. A
later optional section introduces the closed-shell pooling shortcut used by the
function's default.

Start with one full row: both electrons in orbital 0.

In [ ]:
subspace = np.array([config("0001", "0001")])  # just the ground-floor seating

energy, state, occupancies, spin = solve_fermion(
    subspace, hcore, eri, open_shell=True
)
subspace_summary(subspace, open_shell=True, state=state, label="one-row solve")
print(f"one-row total energy: {energy + e_core:.6f} Ha")
print(f"single-determinant reference: {e_hf:.6f} Ha")
print("✅ identical in the Hartree-Fock orbital basis" if np.isclose(energy + e_core, e_hf)
      else "❌ unexpected mismatch")

**In the Hartree-Fock orbital basis, the one-determinant subspace reproduces the
Hartree-Fock energy.** Hartree-Fock did more than pick a row: it first optimized the
orbitals. Once we express the Hamiltonian in those optimized orbitals, its determinant
is the best one-row answer.

Now the reference names have a concrete job:

| Method | Role in this notebook |
|---|---|
| Hartree-Fock (HF) | Best single determinant, including optimized orbitals |
| Full configuration interaction (FCI) | All valid determinants, exact within this orbital basis set |
| `solve_fermion` | Best state in the determinant product space generated by our supplied rows |

> **📏 Units and the yardstick.** Those energies print in **Hartree (Ha)**, the natural
> energy currency of the atomic world: 1 Ha is about twice the energy it takes to rip
> hydrogen's electron away entirely. Chemistry plays for far smaller stakes, so our
> working unit is the **milli-Hartree (mHa)**, one thousandth of a Hartree. The bar
> that matters is **chemical accuracy**: land within **1.6 mHa** (1 kcal/mol) of the
> truth and your predictions about reactions can be trusted. Hartree-Fock misses H₂ by
> ~25 mHa, more than 15x too coarse. The missing piece is called **correlation
> energy**: electrons actively dodging each other, which no single seating chart can
> describe. Capturing it is the central task, and it is a *subspace* task.


In [ ]:
print(f"Hartree-Fock energy : {e_hf:.6f} Ha")
print(f"exact (FCI) energy  : {e_fci:.6f} Ha")
print(f"HF misses by        : {(e_hf - e_fci) * 1000:.2f} milli-Hartree (mHa)")

<details>
<summary>🔎 <b>Under the hood: what the integrals and reference methods mean</b></summary>

To crop-and-diagonalize we need the molecule's Hamiltonian. In notebook 1, $H$ was a
table of numbers small enough to build outright. For a molecule nobody stores the whole
table; instead, chemistry packages hand you a compact **kit of numbers** from which any
entry $\langle \text{config}_i | H | \text{config}_j \rangle$ can be assembled on
demand. The classical package `pyscf` computes that kit from just the atom positions
and a basis-set name:

- `hcore[p, q]`: the **one-electron numbers**: the energy of a lone electron sitting in
  orbital $p$, or hopping between orbitals $p$ and $q$,
- `eri[p, q, r, s]`: the **two-electron numbers**: how strongly two electrons repel,
  depending on which orbitals the pair occupies,
- `e_core`: a constant energy offset (the two nuclei pushing on each other),
- the **Hartree-Fock (HF)** energy: chemistry's standard first approximation. The HF
  procedure finds the single best seating chart, optimizing the orbital shapes along
  the way, and reports that one chart's energy,
- the **FCI** energy (*full configuration interaction*: diagonalize using every valid
  configuration at once): the exact answer within this basis set, affordable only
  because H₂ is tiny. It plays the role `np.linalg.eigh` played in notebook 1: our
  answer key.

Read `hcore` and `eri` as the molecule's version of notebook 1's two rules. The
one-electron numbers are rule one: each electron pays an energy for the orbital it
occupies and can hop between orbitals, so `hcore` plays the role of both the comfort
scores *and* the hopping links from the magnet chain. The two-electron numbers are
rule two, and this one is genuinely new: electrons repel one another, with a cost
that depends on which seats the pair occupies. The architecture is unchanged: a
rulebook whose entries score and connect seating charts; only the rules differ.

One bookkeeping consequence appears in every energy we print. The electrons' rulebook
yields the *electronic* energy; the nuclei's mutual repulsion adds the constant
`e_core` on top. **Total = electronic + `e_core`**, and we always compare totals.

One naming heads-up, because you'll see the word everywhere (including in this lab):
chemists call `hcore` and `eri` the molecule's **integrals**. The name comes from how
each number is calculated: by summing contributions from every point in 3D space where
the electron clouds overlap, which in calculus is an integral. You will never compute
one yourself. `pyscf` does the calculus and hands us plain NumPy arrays; from here on,
"the integrals" just means "the kit of numbers that defines this molecule's $H$".

Treat this cell as boilerplate to copy into any future SQD project. (Distances, by
the way, are in **Angstrom**: a tenth of a nanometer, about the width of a single
atom. H₂'s natural bond length is 0.74 Å.)

</details>

### ✍️ Your turn 1 · add one sample row

Grow the stack to two rows by adding the configuration where **both** electrons jump
from orbital 0 to orbital 1 (the right panel of our seating diagram). The second full
row adds one alpha string and one beta string, so predict both:

- How many full rows are supplied?
- How many determinant basis states will their Cartesian product generate?

Then measure how much of the Hartree-Fock-to-FCI gap the enlarged product space closes.

> **Bit-order reminder:** `β3 β2 β1 β0 | α3 α2 α1 α0`, with orbital 0
> at the right edge of each half. Keep passing separate strings to `config(alpha, beta)`;
> do not concatenate packed rows by hand.

In [ ]:
# ✏️ =============== YOUR CODE HERE ===============
alpha_bumped = ...  # TODO: 4-bit string with the alpha electron in orbital 1
beta_bumped = ...   # TODO: same idea for the beta electron
# ✏️ ============== END OF YOUR CODE ==============
assert alpha_bumped is not Ellipsis, "✏️ Fill in alpha_bumped and beta_bumped above first!"

subspace2 = np.array([
    config("0001", "0001"),             # ground floor
    config(alpha_bumped, beta_bumped),  # both electrons bumped up
])
for row in subspace2:
    assert validate_configuration(row, nelec=(1, 1), verbose=False)

e2, state2, *_ = solve_fermion(subspace2, hcore, eri, open_shell=True)
subspace_summary(subspace2, open_shell=True, state=state2, label="two-row solve")

print(f"Hartree-Fock error:            {(e_hf - e_fci) * 1000:6.2f} mHa")
print(f"two-row product-space error:   {(e2 + e_core - e_fci) * 1000:6.2f} mHa")
assert e2 + e_core <= e_hf + 1e-10, "adding determinant basis states must lower or keep the energy"
print("✅ energy went down, as the variational principle promised")

<details>
<summary>💡 <b>Solution: Your turn 1</b></summary>

```python
alpha_bumped = "0010"
beta_bumped = "0010"
```

You supplied **2 full rows**. They contain 2 distinct alpha strings and 2 distinct
beta strings, so the solver uses a **2 x 2 = 4 determinant** product space. One extra
row closes part of the gap because it adds not only the both-bumped determinant, but
also the two mixed alpha/beta pairings.

At the natural bond length the correlation is spread over several small "dodge moves".
Hold that thought for the finale, where we stretch the bond and this same bumped
configuration becomes comparably important.

</details>

Now compare three nested cases: one supplied row, two supplied rows, and all 16 valid
full rows. The labels report both the input row count and the determinant dimension
actually diagonalized.

In [ ]:
assert "e2" in globals(), "✏️ Run (and finish) Your turn 1 first: this plot needs `e2`."

half_strings = ["0001", "0010", "0100", "1000"]
subspace_all = np.array([config(a, b) for a, b in itertools.product(half_strings, half_strings)])
e_all, state_all, *_ = solve_fermion(subspace_all, hcore, eri, open_shell=True)

levels = [
    (energy + e_core, "1 sample row\n1 determinant", tuple(state.amplitudes.shape)),
    (e2 + e_core, "2 sample rows\n4 determinants", tuple(state2.amplitudes.shape)),
    (e_all + e_core, "16 sample rows\n16 determinants (= FCI)", tuple(state_all.amplitudes.shape)),
]

fig, ax = plt.subplots(figsize=(8, 4.3))
for i, (E, label, shape) in enumerate(levels):
    ax.hlines(E, i - 0.35, i + 0.35, lw=4)
    ax.text(i, E + 0.0012,
            f"{(E - e_fci) * 1000:.2f} mHa above exact\nsolver shape {shape}",
            ha="center", fontsize=8)
ax.axhline(e_fci, color="gray", ls=":", lw=1, label="exact (FCI)")
ax.set_xticks(range(3)), ax.set_xticklabels([label for _, label, _ in levels], fontsize=9)
ax.set_ylabel("total energy (Ha)")
ax.margins(y=0.18)
ax.set_title("sample rows generate determinant product spaces")
ax.legend()
plt.tight_layout()
plt.show()

**How to read the staircase:** each horizontal bar is the best total energy available
inside one determinant product space, and the dotted line is the exact answer. Bars
only ever step *down* as the retained alpha and beta string sets grow. The annotation
on each bar gives the remaining distance from exact and the solver's actual
$M \times N$ amplitude-array shape.

**🧠 Checkpoint 2.** You hand `solve_fermion` two full sample rows:
(`α=0001, β=0001`) and (`α=0010, β=0010`). With `open_shell=True`, how many
alpha strings, beta strings, and determinant basis states does the solver use?

<details>
<summary>💡 <b>Check your answer</b></summary>

There are 2 unique alpha strings and 2 unique beta strings. Their Cartesian product
contains $2 \times 2 = 4$ determinants, including the two mixed pairings you never
supplied as full rows.

</details>

### Optional optimization · closed-shell spin pooling

The default `open_shell=False` assumes alpha and beta play symmetric roles. It merges
the half-string sets, then uses the merged set for both spins. That can enlarge the
subspace beyond the transparent separate-sector construction above.

For example, the second row below introduces a new alpha string but no new beta
string. Keeping sectors separate yields $2 \times 1 = 2$ determinants. Closed-shell
pooling merges the two strings and yields $2 \times 2 = 4$.

In [ ]:
asymmetric_rows = np.array([
    config("0001", "0001"),
    config("0010", "0001"),
])
subspace_summary(asymmetric_rows, open_shell=True, label="open_shell=True: sectors stay separate")
print()
subspace_summary(asymmetric_rows, open_shell=False, label="open_shell=False: closed-shell pooling")

## 4 · Which configurations matter? (chemistry's concentration plot)

In notebook 1 we peeked at the exact magnet ground state and found its weight
concentrated on a few bitstrings. H₂ is small enough to do the same peek: the FCI
calculation gives the exact amplitude of every configuration.

In [ ]:
# the four possible half-strings ('0001', '0010', '0100', '1000') in pyscf's order:
half_labels = [f"{s:04b}" for s in pyscf.fci.cistring.make_strings(range(4), 1)]
weights = np.abs(civec) ** 2  # valid for real or complex amplitudes

flat = [(weights[i, j], f"α:{a} β:{b}")
        for (i, a), (j, b) in itertools.product(enumerate(half_labels), enumerate(half_labels))]
flat.sort(reverse=True)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(range(16), [w for w, _ in flat], color="#4589ff")
ax.set_yscale("log")
ax.set_xticks(range(16))
ax.set_xticklabels([lbl for _, lbl in flat], rotation=90, fontsize=8, family="monospace")
ax.set_ylabel("weight $|c|^2$ (log)")
ax.set_title("H2's exact ground state, configuration by configuration")
plt.tight_layout()
plt.show()

The same silhouette as the magnet: one dominant tower (Hartree-Fock), a couple of
important supporters, and a long tail of near-irrelevance.

**🧠 Checkpoint 3.** In notebook 1 the towers were `00000000` and `11111111`. Here the
tower is the HF configuration. What plays the role of "basis" in each case, and what
property of the ground state makes the SQD strategy viable in both?

<details>
<summary>💡 <b>Check your answer</b></summary>

For the magnet, the basis was "spin up/down per site"; for the molecule it's "electron
seating charts". In both, the ground state is **concentrated**: a short list of basis
states carries almost all the weight. Concentration is the property SQD exploits,
and molecular ground states exhibit it strongly because one seating chart (HF) already
describes most of the physics.

</details>

So let's replay notebook 1's convergence experiment, chemistry edition: grow the
subspace from the most important configuration downward and watch the error crash
through the chemical-accuracy line.

Grow the input from the most important full rows downward. The horizontal axis is now
labeled honestly as **sample rows supplied**. Every point is annotated with the
actual determinant product dimension produced by the accumulated alpha and beta
strings.

In [ ]:
ranked = np.argsort(weights.ravel())[::-1]
order = [(k // 4, k % 4) for k in ranked]

sample_row_counts = range(1, 17)
errors, dimensions, shapes = [], [], []
for k in sample_row_counts:
    rows = np.array([config(half_labels[i], half_labels[j]) for i, j in order[:k]])
    e_k, state_k, *_ = solve_fermion(rows, hcore, eri, open_shell=True)
    errors.append((e_k + e_core - e_fci) * 1000)
    shape = tuple(int(x) for x in state_k.amplitudes.shape)
    shapes.append(shape)
    dimensions.append(int(np.prod(shape)))

fig, ax = plt.subplots(figsize=(8.2, 4.4))
ax.plot(list(sample_row_counts), errors, "o-", color="#fa4d56")
ax.axhline(1.6, color="green", ls="--", label="chemical accuracy (1.6 mHa)")
for k, error, shape in zip(sample_row_counts, errors, shapes):
    if k <= 6 or k in (8, 12, 16):
        ax.annotate(f"{shape[0]}x{shape[1]}", (k, max(error, 1e-8)),
                    textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7)
ax.set_xlabel("full sample rows supplied (most important first)")
ax.set_ylabel("energy error vs exact (mHa)")
ax.set_title("row count and determinant dimension are not the same")
ax.legend()
plt.tight_layout()
plt.show()

print("rows -> determinant dimension:", dict(zip(sample_row_counts, dimensions)))

The energy becomes exact once the accumulated rows have supplied all four alpha
strings and all four beta strings, because their product then spans all 16 valid
determinants. The important count is the solver's product-space dimension, not merely
the number of full rows that happened to introduce those half-strings.

## 5 · Rehearsal: sampling instead of peeking

Peeking at exact amplitudes is a tiny-system luxury. Use one pretend-perfect sample of
50 shots to transfer notebook 1's idea into the molecular representation. This is a
single rehearsal rather than another full convergence study.

In [ ]:
rehearsal_shots = 50
draws = rng.choice(16, size=rehearsal_shots, p=weights.ravel() / weights.sum())
frequency_table = dict(zip(*np.unique(draws, return_counts=True)))
seen = np.array(sorted(frequency_table))
rows_sampled = np.array([
    config(half_labels[index // 4], half_labels[index % 4])
    for index in seen
])

e_sampled, state_sampled, *_ = solve_fermion(
    rows_sampled, hcore, eri, open_shell=True
)
subspace_summary(rows_sampled, open_shell=True, state=state_sampled,
                 label=f"support found in {rehearsal_shots} exact-state shots")
print(f"energy error: {(e_sampled + e_core - e_fci) * 1000:.3f} mHa")
print("sample frequencies by flattened configuration index:", frequency_table)

Repeated shots mostly land on configurations we already have; that redundancy is the
price of not knowing the ranking in advance, and it's cheap.

For this direct `solve_fermion` call, duplicate shots do not create duplicate basis
vectors, so only the unique support enters the projection. Keep the frequency table,
though: notebook 3's configuration-recovery loop will use those frequencies to infer
average orbital occupancies before it constructs each subspace.

## 6 · Finale: breaking a bond

Why obsess over a few mHa? Because sometimes Hartree-Fock isn't 25 mHa wrong, it's
*catastrophically* wrong. Pull the two H atoms apart: each atom must end up with its own
electron, but HF's single seating chart forces the electrons to keep sharing orbitals,
and its energy soars.

First compare three static snapshots of the exact configuration weights: a short bond,
the natural bond length, and a stretched bond. The two named configurations are
labeled directly so the comparison does not depend on color alone. Every molecular
calculation is cached once and reused by the energy-curve exercise below.

In [ ]:
def bond_key(distance):
    """Stable two-decimal cache key for a bond distance."""
    return round(float(distance), 2)


bond_lengths = np.round(np.linspace(0.5, 2.5, 11), 2)
snapshot_distances = [0.5, 0.74, 2.5]
all_distances = sorted({bond_key(d) for d in bond_lengths} | {bond_key(d) for d in snapshot_distances})
molecule_cache = {distance: h2_molecule(distance) for distance in all_distances}

pair_labels = [f"α:{a} β:{b}" for a, b in itertools.product(half_labels, half_labels)]
order0 = np.argsort(weights.ravel())[::-1]
labels0 = [pair_labels[k] for k in order0]

hf_label = "α:0001 β:0001"
bumped_label = "α:0010 β:0010"

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
for ax, distance in zip(axes, snapshot_distances):
    weights_d = np.abs(molecule_cache[bond_key(distance)][5]) ** 2
    values = np.maximum(weights_d.ravel()[order0], 1e-12)
    colors = ["#8a3ffc" if label == hf_label
              else "#fa4d56" if label == bumped_label
              else "#4589ff" for label in labels0]
    bars = ax.bar(range(16), values, color=colors)
    ax.set_yscale("log")
    ax.set_ylim(1e-7, 1.8)
    ax.set_xticks(range(16))
    ax.set_xticklabels(labels0, rotation=90, fontsize=6.5, family="monospace")
    ax.set_title(f"bond distance {distance:.2f} Å")
    ax.set_ylabel("weight $|c|^2$ (log)")
    for target, text in [(hf_label, "HF"), (bumped_label, "both bumped")]:
        x = labels0.index(target)
        y = values[x]
        ax.annotate(text, (x, y), textcoords="offset points", xytext=(0, 6),
                    ha="center", fontsize=8, fontweight="bold")
plt.show()

Watch what a **2-row input producing a 4-determinant product space** does across the
whole curve. The cached dictionary supplies each distance's `hcore`, `eri`, `e_core`,
HF energy, FCI energy, and exact state without recomputing the chemistry.

> **Bit-order reminder:** `β3 β2 β1 β0 | α3 α2 α1 α0`, with orbital 0
> at the right edge of each half. Keep passing separate strings to `config(alpha, beta)`;
> do not concatenate packed rows by hand.

### ✍️ Your turn 2 · solve the product space at every distance

Build the same 2-row stack as Your turn 1 once, before the loop, then call
`solve_fermion(..., open_shell=True)` with each distance's cached integrals.

In [ ]:
curve_hf, curve_fci, curve_sub = [], [], []

# ✏️ =============== YOUR CODE HERE ===============
rows = ...  # TODO: the same 2-row stack as Your turn 1
# ✏️ ============== END OF YOUR CODE ==============
assert rows is not Ellipsis, "✏️ Build `rows` first: the same 2-row stack as Your turn 1."

for distance in bond_lengths:
    e_hf_d, hcore_d, eri_d, e_core_d, e_fci_d, _ = molecule_cache[bond_key(distance)]
    # ✏️ =============== YOUR CODE HERE ===============
    e_sub_d, *_ = solve_fermion(..., ..., ..., open_shell=True)
    # ✏️ ============== END OF YOUR CODE ==============
    curve_hf.append(e_hf_d)
    curve_fci.append(e_fci_d)
    curve_sub.append(e_sub_d + e_core_d)

curve_hf = np.asarray(curve_hf)
curve_fci = np.asarray(curve_fci)
curve_sub = np.asarray(curve_sub)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(bond_lengths, curve_hf, "o--", color="#8a3ffc", label="Hartree-Fock (1 determinant)")
ax1.plot(bond_lengths, curve_sub, "s-", color="#4589ff", label="2 rows -> 4 determinants")
ax1.plot(bond_lengths, curve_fci, "-", color="black", lw=1, label="exact (FCI)")
ax1.set_xlabel("H-H distance (Angstrom)"), ax1.set_ylabel("energy (Ha)")
ax1.set_title("breaking the H-H bond")
ax1.legend()

ax2.semilogy(bond_lengths, (curve_hf - curve_fci) * 1000, "o--", color="#8a3ffc")
ax2.semilogy(bond_lengths, (curve_sub - curve_fci) * 1000, "s-", color="#4589ff")
ax2.axhline(1.6, color="green", ls="--", label="chemical accuracy")
ax2.set_xlabel("H-H distance (Angstrom)"), ax2.set_ylabel("error vs exact (mHa, log)")
ax2.set_title("HF error explodes; the product space stays calmer")
ax2.legend()
plt.tight_layout()
plt.show()

print(f"error at 2.5 Angstrom: HF {(curve_hf[-1] - curve_fci[-1]) * 1000:8.1f} mHa"
      f" | 4-determinant space {(curve_sub[-1] - curve_fci[-1]) * 1000:.1f} mHa")

**How to read these charts:** the left panel shows absolute energies at each distance
(lower is better); the right panel shows the same data as errors above exact, on a
log scale, against the chemical-accuracy line.

As the bond stretches, Hartree-Fock's error balloons while the small determinant
product space remains much closer. Its remaining error comes from determinants whose
alpha or beta half-strings never entered the two-row input.

<details>
<summary>💡 <b>Solution: Your turn 2</b></summary>

```python
rows = np.array([
    config("0001", "0001"),
    config("0010", "0010"),
])

# inside the loop:
e_sub_d, *_ = solve_fermion(rows, hcore_d, eri_d, open_shell=True)
```

Remember to add `e_core_d` when comparing totals; the plotting code already does.

</details>

**🧠 Checkpoint 4.** At large distance, the "both bumped up" configuration becomes almost
as heavy as HF itself. Why does pulling the atoms apart *require* a second configuration?

<details>
<summary>💡 <b>Check your answer</b></summary>

Far apart, the true state is "one electron on the left atom, one on the right". In the
molecule's orbital language, that localized picture can only be built by **mixing** the
ground-floor and both-bumped configurations, together with the mixed-spin pairings in
the Cartesian product. One seating chart alone always leaves some probability of both
electrons crowding one atom, which costs energy. This is why bond-breaking is the
classic stress test where one-seating-chart methods like Hartree-Fock fail and
subspace methods shine.

</details>

**🧠 Final checkpoint.** Distinguish these four objects in one sentence each:

1. a supplied full sample row,
2. an alpha string,
3. a beta string,
4. the determinant product space diagonalized by `solve_fermion`.

<details>
<summary>💡 <b>Check your answer</b></summary>

A full sample row is one packed `beta + alpha` bitstring. Its right half supplies one
alpha occupation string and its left half supplies one beta occupation string. The
solver deduplicates the retained alpha and beta sets separately, then forms every pair;
if their sizes are $M$ and $N$, the determinant basis has dimension $M \times N$.

</details>

## 🔁 Recap: the SQD recipe, version 0.2

1. **Molecule → integrals** (`pyscf` boilerplate: `hcore`, `eri`, `e_core`).
2. **Collect full configuration rows**, while tracking their alpha and beta halves.
3. **Construct the determinant product space**: $M$ alpha strings times $N$ beta strings.
4. **`solve_fermion(..., open_shell=True)`** projects and diagonalizes transparently in that space.
5. **Add `e_core`**, compare against references, and grow the spin-string sets until the energy converges.

And the dictionary from the top of the notebook, now filled in:

| Notebook 1 | Notebook 2 |
|---|---|
| Bitstring of magnets | Packed electron configuration row |
| Basis-state shortlist | Alpha/beta string sets and their determinant Cartesian product |
| Crop + `np.linalg.eigh` | `solve_fermion`, plus `e_core` |
| Concentration: two wells and ripples | Concentration: the HF tower and its supporters |
| Unique support is enough for a direct projection | Frequencies are preserved for notebook 3's recovery loop |

What's still fake: the rehearsal sampled from the exact answer, which is exactly the
thing we can't have for real molecules. **Notebook 3** replaces that peek with a
chemistry-informed quantum circuit, controlled noise experiments, and configuration
recovery.